In [33]:
import pandas as pd
import numpy as np
import sys   
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # add project root so src is importable

from src.config import (
    CUSTOMERS_TRAIN, LOANS_CLEAN, TRANSACTIONS_CLEAN, TRAIN_IDS, CHURN_FEATURES,DEFAULT_FEATURES,SEGMENT_FEATURES,RANDOM_STATE,
) 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score ,average_precision_score 
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier 

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder  

In [34]:
churn_model=pd.read_parquet(CHURN_FEATURES)
churn_model

,customer_id,age,region,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,smartphone_user,complaints_12m,failed_txns_12m,...,credit_score,churned_12m,age_missing,income_band_missing,total_counts,total_amount,active_months,first,last,difference
0,C100000,33.0,Sindh,13.0,25-50k,31700.0,1,1,0,1,...,442,N,0,0,19.0,33630.0,4.0,15.0,4.0,-11.0
1,C100002,41.0,KP,13.0,25-50k,41000.0,1,1,0,2,...,426,N,0,0,21.0,41040.0,6.0,8.0,13.0,5.0
2,C100003,23.0,Punjab,15.0,25-50k,15800.0,1,1,1,0,...,399,N,0,1,46.0,88670.0,3.0,46.0,0.0,-46.0
3,C100006,32.0,Punjab,21.0,50-100k,45500.0,2,1,0,3,...,489,N,0,0,12.0,20760.0,3.0,4.0,8.0,4.0
4,C100007,25.0,Balochistan,17.0,50-100k,48900.0,1,1,1,0,...,451,N,0,0,8.0,29560.0,4.0,7.0,1.0,-6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11755,C114994,26.0,Punjab,21.0,<25k,25600.0,0,1,0,1,...,499,N,0,0,22.0,39100.0,6.0,9.0,13.0,4.0
11756,C114995,44.0,Punjab,15.0,25-50k,24500.0,2,1,0,2,...,460,N,0,0,12.0,22840.0,6.0,8.0,4.0,-4.0
11757,C114996,43.0,Punjab,57.0,<25k,19700.0,2,1,0,2,...,525,N,0,0,12.0,17530.0,4.0,6.0,6.0,0.0
11758,C114998,34.0,Sindh,13.0,25-50k,37400.0,0,0,2,0,...,370,N,0,0,29.0,57290.0,6.0,11.0,18.0,7.0


In [35]:
churn_model["churned_12m"]=churn_model["churned_12m"].map({"Y":1,"N":0})

In [36]:
churn_model["churned_12m"].value_counts(normalize=True)

churned_12m
0    0.922959
1    0.077041
Name: proportion, dtype: float64

In [37]:
X=churn_model.drop(columns=["first", "last", "customer_id", "churned_12m"])
y=churn_model["churned_12m"] 

In [38]:
X.dtypes

age                        float64
region                    category
wallet_tenure_months       float64
declared_income_band      category
avg_monthly_inflow_pkr     float64
dependents                   int64
smartphone_user              int64
complaints_12m               int64
failed_txns_12m              int64
has_savings                  int64
savings_balance_pkr        float64
has_insurance                int64
credit_score                 int64
age_missing                  int64
income_band_missing          int64
total_counts               float64
total_amount               float64
active_months              float64
difference                 float64
dtype: object

In [39]:
X.shape

(11760, 19)

In [40]:
X[["total_counts", "total_amount", "active_months", "difference"]].corr()

,total_counts,total_amount,active_months,difference
total_counts,1.000000,0.943384,0.280162,-0.019522
total_amount,0.943384,1.000000,0.268407,-0.013988
active_months,0.280162,0.268407,1.000000,0.119662
difference,-0.019522,-0.013988,0.119662,1.000000


In [41]:
X=X.drop(columns=["total_amount"])
X

,age,region,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,smartphone_user,complaints_12m,failed_txns_12m,has_savings,savings_balance_pkr,has_insurance,credit_score,age_missing,income_band_missing,total_counts,active_months,difference
0,33.0,Sindh,13.0,25-50k,31700.0,1,1,0,1,0,0.0,0,442,0,0,19.0,4.0,-11.0
1,41.0,KP,13.0,25-50k,41000.0,1,1,0,2,0,0.0,1,426,0,0,21.0,6.0,5.0
2,23.0,Punjab,15.0,25-50k,15800.0,1,1,1,0,1,10000.0,0,399,0,1,46.0,3.0,-46.0
3,32.0,Punjab,21.0,50-100k,45500.0,2,1,0,3,1,13000.0,0,489,0,0,12.0,3.0,4.0
4,25.0,Balochistan,17.0,50-100k,48900.0,1,1,1,0,1,19900.0,0,451,0,0,8.0,4.0,-6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11755,26.0,Punjab,21.0,<25k,25600.0,0,1,0,1,1,69700.0,1,499,0,0,22.0,6.0,4.0
11756,44.0,Punjab,15.0,25-50k,24500.0,2,1,0,2,1,12500.0,1,460,0,0,12.0,6.0,-4.0
11757,43.0,Punjab,57.0,<25k,19700.0,2,1,0,2,1,23600.0,0,525,0,0,12.0,4.0,0.0
11758,34.0,Sindh,13.0,25-50k,37400.0,0,0,2,0,0,0.0,0,370,0,0,29.0,6.0,7.0


In [42]:
X_train, X_val ,y_train, y_val=train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE,stratify=y
)

In [43]:
print(X_train.shape)
print(X_val.shape)
print(y_train.mean())
print(y_val.mean()) 

(9408, 18)
(2352, 18)
0.07706207482993198
0.07695578231292517


In [44]:
num_cols = X_train.select_dtypes(exclude="category").columns.tolist()
num_cols
#num_cols= ['amount_pkr','term_months','inflow_to_loan_ratio','months_available','average_txns_per_mon','active_ratio','age','wallet_tenure_months','avg_monthly_inflow_pkr','dependents','smartphone_user','complaints_12m','failed_txns_12m','has_savings','savings_balance_pkr','has_insurance','credit_score','age_missing','income_band_missing']

['age',
 'wallet_tenure_months',
 'avg_monthly_inflow_pkr',
 'dependents',
 'smartphone_user',
 'complaints_12m',
 'failed_txns_12m',
 'has_savings',
 'savings_balance_pkr',
 'has_insurance',
 'credit_score',
 'age_missing',
 'income_band_missing',
 'total_counts',
 'active_months',
 'difference']

In [45]:
len(num_cols)

16

In [46]:
nominal_cols = ["region"]

ordinal_cols = ["declared_income_band"]

num_cols= X_train.select_dtypes(exclude="category").columns.tolist() 

income_order = [["<25k", "25-50k", "50-100k", "100-250k", "250k+"]]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("nom", OneHotEncoder(drop="first", handle_unknown="ignore"), nominal_cols),
        ("ord", OrdinalEncoder(categories=income_order), ordinal_cols),
    ],
    remainder="drop",
)

pipe = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])

pipe.fit(X_train, y_train)
b_pred  = pipe.predict(X_val)
b_proba = pipe.predict_proba(X_val)[:, 1]



In [47]:
print(classification_report(y_val, b_pred))
print(roc_auc_score(y_val, b_proba))
print(average_precision_score(y_val, b_proba))

              precision    recall  f1-score   support

           0       0.96      0.62      0.75      2171
           1       0.13      0.68      0.22       181

    accuracy                           0.62      2352
   macro avg       0.54      0.65      0.49      2352
weighted avg       0.89      0.62      0.71      2352

0.7094574132652672
0.1716181869309323


In [48]:
weights = pd.Series(
    pipe.named_steps["model"].coef_[0],
    index=pipe.named_steps["prep"].get_feature_names_out(),
)
weights.sort_values(key=abs, ascending=False)

num__wallet_tenure_months     -0.746264
num__total_counts             -0.550200
num__complaints_12m            0.351193
num__difference               -0.291687
nom__region_Sindh             -0.211237
nom__region_Balochistan       -0.211228
nom__region_KP                -0.185617
num__failed_txns_12m           0.168449
num__has_insurance             0.126499
num__active_months            -0.119989
num__has_savings               0.119133
num__age                      -0.114783
num__avg_monthly_inflow_pkr   -0.103771
num__credit_score              0.101170
nom__region_Islamabad         -0.094460
ord__declared_income_band      0.058177
num__income_band_missing      -0.050731
num__smartphone_user          -0.037291
num__dependents                0.036347
num__savings_balance_pkr       0.028050
num__age_missing              -0.007899
nom__region_Punjab             0.000905
dtype: float64

In [49]:
nominal_cols = ["region"]

ordinal_cols = ["declared_income_band"]

num_cols= X_train.select_dtypes(exclude="category").columns.tolist() 

income_order = [["<25k", "25-50k", "50-100k", "100-250k", "250k+"]]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("nom", OneHotEncoder(drop="first", handle_unknown="ignore"), nominal_cols),
        ("ord", OrdinalEncoder(categories=income_order), ordinal_cols),
    ],
    remainder="drop",
)

pipe_boost = Pipeline([
    ("prep", preprocess),
    ("model", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE)),
])

pipe_boost.fit(X_train, y_train)
b_pred_boost  = pipe_boost.predict(X_val)
b_proba_boost = pipe_boost.predict_proba(X_val)[:, 1]



In [50]:
print(classification_report(y_val, b_pred_boost))
print(roc_auc_score(y_val, b_proba_boost))
print(average_precision_score(y_val, b_proba_boost))

              precision    recall  f1-score   support

           0       0.92      0.99      0.96      2171
           1       0.22      0.03      0.05       181

    accuracy                           0.92      2352
   macro avg       0.57      0.51      0.50      2352
weighted avg       0.87      0.92      0.89      2352

0.6751605161966759
0.13252037733602273


In [51]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_grid = {
    "model__max_depth": [1,2, 3, 4, 5, 6, 8],
    "model__learning_rate": [0.005,0.01, 0.03, 0.05, 0.1, 0.2],
    "model__n_estimators": [100, 200, 300, 500, 800],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__scale_pos_weight": [4,5,7,9,12,14]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=pipe_boost,
    param_distributions=param_grid,
    n_iter=30,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)

search.fit(X_train, y_train)

print(search.best_params_)
print(search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
{'model__subsample': 0.6, 'model__scale_pos_weight': 9, 'model__n_estimators': 100, 'model__max_depth': 4, 'model__learning_rate': 0.05}
0.1909044711660897


In [52]:
best = search.best_estimator_
b_pred_tuned  = best.predict(X_val)
b_proba_tuned = best.predict_proba(X_val)[:, 1]

In [53]:
print(classification_report(y_val, b_pred_tuned))
print(roc_auc_score(y_val, b_proba_tuned))
print(average_precision_score(y_val, b_proba_tuned))

              precision    recall  f1-score   support

           0       0.95      0.79      0.86      2171
           1       0.16      0.48      0.23       181

    accuracy                           0.76      2352
   macro avg       0.55      0.63      0.55      2352
weighted avg       0.89      0.76      0.81      2352

0.7104244549574884
0.16361083980041902


In [54]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Get raw probabilities for Class 1 (defaulters) using your validation data
y_probs = pipe.predict_proba(X_val)[:, 1]

# 2. Test different thresholds to see the exact trade-offs
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5,0.6,0.7,0.8,0.9]
results = []

for t in thresholds:
    y_pred_custom = (y_probs >= t).astype(int)
    results.append({
        "Threshold": t,
        "Good Customer Recall (Class 0)": recall_score(y_val, y_pred_custom, pos_label=0),
        "Churner Precision (Class 1)": precision_score(y_val, y_pred_custom, pos_label=1),
        "Churner Recall (Class 1)": recall_score(y_val, y_pred_custom, pos_label=1),
        "Churner F1-Score": f1_score(y_val, y_pred_custom, pos_label=1),
        "Share Targeted": y_pred_custom.mean()  
    })

# 3. Print the scannable decision matrix
print("\n--- Threshold Decision Matrix ---")
print(pd.DataFrame(results).to_string(index=False))



--- Threshold Decision Matrix ---
 Threshold  Good Customer Recall (Class 0)  Churner Precision (Class 1)  Churner Recall (Class 1)  Churner F1-Score  Share Targeted
      0.30                        0.296177                     0.100118                  0.939227          0.180947        0.721939
      0.35                        0.369415                     0.104058                  0.878453          0.186074        0.649660
      0.40                        0.446338                     0.110289                  0.823204          0.194517        0.574405
      0.45                        0.528789                     0.118103                  0.756906          0.204325        0.493197
      0.50                        0.619991                     0.129747                  0.679558          0.217892        0.403061
      0.60                        0.799632                     0.160232                  0.458564          0.237482        0.220238
      0.70                        0.92077